# Setup & Configuration

In [2]:
# ─── Standard Library ───
import os
import re
import json
import time
import base64
from io import BytesIO
from pathlib import Path
import csv
import ast
import numpy as np

# ─── Third-Party ───
import requests
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from PIL import Image
import Levenshtein

### Configurations

In [3]:
# ─── Retry budget for every API call ───
MAX_RETRIES = 3

# ─── OpenRouter credentials ───
# ─── Load credentials from .env ───
def load_env_key(key_name, default=None):
    env_path = Path(".env")
    if env_path.exists():
        with open(env_path, "r") as f:
            for line in f:
                if line.strip().startswith(f"{key_name}="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    return os.environ.get(key_name, default)

OPENROUTER_API_KEY = load_env_key("OPENROUTER_API_KEY")
# OPENROUTER_API_KEY = load_env_key("OPENROUTER_API_KEY_test") ## Test Key

# MODEL_NAME = "qwen/qwen3-vl-8b-instruct" #Done
# MODEL_NAME = "openai/gpt-4o-mini" #Done
# MODEL_NAME = "rekaai/reka-edge" # Done , remove 23696* 260 = 6,160,960 => $ 0.616096
# MODEL_NAME = "mistralai/ministral-8b-2512" #Done
# MODEL_NAME = "google/gemini-3.1-flash-lite-preview" #Done
# MODEL_NAME = "meta-llama/llama-3.2-11b-vision-instruct" #if needed
       

# ─── Dataset Paths ───
CHARTQA_PATH       = Path("../Datasets/chartqa.parquet")
AR_CHARTQA_PATH    = Path("../Datasets/chartqa_arabic.parquet")
MATHVISION_PATH    = Path("../Datasets/mathvision.parquet")
AR_MATHVISION_PATH = Path("../Datasets/mathvision_arabic.parquet")
SCREENQA_PATH      = Path("../Datasets/screenqa.parquet")
TURTLEBENCH_PATH   = Path("../Datasets/TurtleBench")
PEARL_PATH         = Path("../Datasets/pearl.parquet")
JEEM_PATH          = Path("../Datasets/jeem.parquet")

### Helper Functions

In [4]:
def encode_image_bytes_to_base64(byte_data, resize=None):
    """Convert raw image bytes to a base64 data-URL string.

    Parameters
    ----------
    byte_data : bytes
        Raw image bytes (e.g. from a parquet column).
    resize : tuple[int, int] | None
        Optional (width, height) cap; the image is thumbnailed to fit.

    Returns
    -------
    str | None
        A 'data:image/jpeg;base64,...' string, or None on failure.
    """
    try:
        img = Image.open(BytesIO(byte_data))
        if resize:
            img.thumbnail(resize)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None


def encode_raw_bytes_to_base64(byte_data):
    """Encode raw bytes directly to a base64 data-URL (no PIL processing)."""
    try:
        b64 = base64.b64encode(byte_data).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None


def encode_image_file_to_base64(image_path, include_prefix=True):
    """Open an image file from disk, convert to JPEG base64.

    Parameters
    ----------
    image_path : str | Path
        Path to the image on disk.
    include_prefix : bool
        If True, return with 'data:image/jpeg;base64,' prefix.

    Returns
    -------
    str | None
    """
    try:
        with Image.open(image_path) as img:
            if img.mode != "RGB":
                img = img.convert("RGB")
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=85)
            b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
            return f"data:image/jpeg;base64,{b64}" if include_prefix else b64
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None


def query_openrouter(prompt, base64_image, model_name=None, max_retries=MAX_RETRIES):
    """Send an image + prompt to OpenRouter and return (response_text, duration)."""
    model = model_name or MODEL_NAME
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    # ── OpenAI vision token billing ────────────────────────────────────
    # OpenAI applies a ~33× billing multiplier to gpt-4o-mini image
    # tokens. Using detail:"low" forces a fixed 2833 billing tokens per
    # image instead of ~25 000 (detail:"high" or "auto").
    # This does NOT degrade quality for chart/text images.
    if model == "openai/gpt-4o-mini":
        image_block = {
            "type": "image_url",
            "image_url": {"url": base64_image, "detail": "low"},
        }
    else:
        image_block = {
            "type": "image_url",
            "image_url": {"url": base64_image},
        }

    payload = {
        "model": model,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                image_block,
            ],
        }],
        "temperature": 0.1,
    }

    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=120,
            )
            # This line triggers an HTTPError if the status code is 4xx or 5xx
            resp.raise_for_status() 
            
            text = resp.json()["choices"][0]["message"]["content"].strip()
            return text, time.time() - start
            
        except requests.exceptions.Timeout:
            wait_time = 5 * (attempt + 1)
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: timed out.")
            
        except requests.exceptions.RequestException as e:
            # Check specifically if the error is a 429 Rate Limit
            if getattr(e.response, 'status_code', None) == 429:
                # Apply a heavier backoff penalty specifically for rate limits
                wait_time = 10 * (attempt + 1) 
                print(f"   🛑 Rate limited (429)! Backing off for {wait_time}s...")
            else:
                wait_time = 5 * (attempt + 1)
                print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
                
        # Sleep before trying again
        if attempt < max_retries - 1:
            time.sleep(wait_time)

    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def query_openrouter_text(prompt, model_name=None, max_retries=MAX_RETRIES):
    """Send a text-only prompt to OpenRouter and return (response_text, duration)."""
    model = model_name or MODEL_NAME
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
    }

    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=120,
            )
            resp.raise_for_status()
            text = resp.json()["choices"][0]["message"]["content"].strip()
            return text, time.time() - start
        except requests.exceptions.Timeout:
            wait_time = 5 * (attempt + 1)
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: timed out.")
        except requests.exceptions.RequestException as e:
            if getattr(e.response, 'status_code', None) == 429:
                wait_time = 10 * (attempt + 1)
                print(f"   🛑 Rate limited (429)! Backing off for {wait_time}s...")
            else:
                wait_time = 5 * (attempt + 1)
                print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
        if attempt < max_retries - 1:
            time.sleep(wait_time)
    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def load_progress(output_csv, key_columns):
    """Load already-processed task keys from an existing CSV to enable resuming.

    Parameters
    ----------
    output_csv : Path
        CSV file path.
    key_columns : list[str]
        Column(s) whose concatenation forms a unique task key.

    Returns
    -------
    tuple[set, bool]
        (processed_keys, file_exists)
    """
    processed = set()
    exists = output_csv.is_file()
    if exists:
        try:
            df = pd.read_csv(output_csv, usecols=lambda c: c in key_columns, dtype=str, encoding="utf-8-sig")
            if not df.empty and all(c in df.columns for c in key_columns):
                if len(key_columns) == 1:
                    processed = set(df[key_columns[0]])
                else:
                    processed = set(
                        df[key_columns[0]].str.cat(
                            [df[c] for c in key_columns[1:]], sep="_"
                        )
                    )
                print(f"🔄 Resuming — skipping {len(processed)} already-processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV: {e}")
    return processed, exists


def append_result(result_dict, output_csv, file_exists):
    """Append a single result row to CSV, writing the header only once.

    Returns
    -------
    bool
        Updated file_exists flag (always True after first write).
    """
    # Use utf-8-sig (adds BOM) for new files so Excel recognises Arabic;
    # plain utf-8 for appends to avoid extra BOMs mid-file.
    enc = "utf-8-sig" if not file_exists else "utf-8"
    with open(output_csv, "a", newline="", encoding=enc) as f:
        writer = csv.DictWriter(f, fieldnames=result_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(result_dict)
    return True


def safe_model_filename(name):
    """Sanitise a model name for use inside a filename."""
    return name.replace("/", "_").replace(":", "_")


def extract_image_bytes(row):
    """Extract raw image bytes from a parquet row (handles nested dict or flat column)."""
    if "decoded_image" in row and isinstance(row["decoded_image"], dict):
        return row["decoded_image"].get("bytes")
    if "image" in row and isinstance(row["image"], dict) and "bytes" in row["image"]:
        return row["image"]["bytes"]
    if "image" in row and isinstance(row["image"], bytes):
        return row["image"]
    return row.get("bytes")

# MathVision

## Original

### Setup and Prompt

In [ ]:
# ─── Load MathVision dataset ───
mathvision_df = None
if MATHVISION_PATH.exists():
    mathvision_df = pd.read_parquet(MATHVISION_PATH)
    print(f"✅ Loaded {len(mathvision_df)} rows.")
else:
    print(f"❌ Dataset not found at {MATHVISION_PATH}.")

MATHVISION_OPENROUTER_LIMIT = None  # Set to None for full evaluation

MATHVISION_SYSTEM_PROMPT = """
Answer using only the image and text. 
RULES:
- Output EXACT final answer ONLY.
- NO explanations, reasoning, or filler words.
- If it is a Multiple choice question: output the letter only (A, B, C, D,...).
- Math: output numbers or equations directly.
"""

### Run

In [ ]:
def run_mathvision_openrouter(df, limit=None):
    """Run MathVision evaluation through OpenRouter backend."""
    print(f"\n🚀 MathVision OpenRouter — {MODEL_NAME}...")

    results_dir = Path("MathVisionResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mathvision_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["id"])

    count = 0
    for _, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break
        task_id = str(row["id"])
        if task_id in processed:
            continue

        level   = str(row.get("level", "N/A"))
        subject = str(row.get("subject", "N/A"))
        query   = str(row["question"])
        gt      = str(row["answer"]).strip()

        print(f"\nTask {task_id} | {subject} | Lvl {level}")

        img_bytes = extract_image_bytes(row)
        b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{MATHVISION_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_math(answer, gt)

        file_exists = append_result({
            "id": task_id, "level": level, "subject": subject,
            "image": str(row.get("image", "N/A")), "question": query,
            "ground_truth": gt, "model_answer": answer,
            "evaluation_passed": passed, "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_id)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
        time.sleep(1)

if mathvision_df is not None:
    run_mathvision_openrouter(mathvision_df, limit=MATHVISION_OPENROUTER_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")

## Arabic

### Setup and Prompt

In [ ]:
# ─── Load MathVision dataset ───
mathvision_df = None
if AR_MATHVISION_PATH.exists():
    mathvision_df = pd.read_parquet(AR_MATHVISION_PATH)
    print(f"✅ Loaded {len(mathvision_df)} rows.")
else:
    print(f"❌ Dataset not found at {AR_MATHVISION_PATH}.")

AR_MATHVISION_OPENROUTER_LIMIT = None  # Set to None for full evaluation

AR_MATHVISION_SYSTEM_PROMPT = """
أجب باستخدام الصورة والنص فقط.
قواعد:
- قدم الإجابة النهائية فقط.
- بدون شروحات، تبريرات، أو حشو.
- اذا كان السؤال ذو خيارات متعددة: اكتب الحرف فقط (A, B, C, D).
- الرياضيات: اكتب الأرقام أو المعادلات مباشرة.
"""

### Run

In [ ]:
def run_mathvision_openrouter(df, limit=None):
    """Run Arabic MathVision evaluation through OpenRouter backend."""
    print(f"\n🚀 Arabic MathVision OpenRouter — {MODEL_NAME}...")

    results_dir = Path("MathVisionResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mathvisionarabic_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["id"])

    count = 0
    for _, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break
        task_id = str(row["id"])
        if task_id in processed:
            continue

        level   = str(row.get("level", "N/A"))
        subject = str(row.get("subject", "N/A"))
        query   = str(row["arabic_question"])
        gt      = str(row["answer"]).strip()

        print(f"\nTask {task_id} | {subject} | Lvl {level}")

        img_bytes = extract_image_bytes(row)
        b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{AR_MATHVISION_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_math(answer, gt)

        file_exists = append_result({
            "id": task_id, "level": level, "subject": subject,
            "image": str(row.get("image", "N/A")), "question": query,
            "ground_truth": gt, "model_answer": answer,
            "evaluation_passed": passed, "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_id)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
        time.sleep(1)

if mathvision_df is not None:
    run_mathvision_openrouter(mathvision_df, limit=AR_MATHVISION_OPENROUTER_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")

# ChartQA

In [ ]:
def normalize_arabic_text(text):
    """Converts Eastern Arabic numerals to Western, and normalizes Arabic punctuation."""
    arabic_to_western = {
        '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', 
        '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9',
        '٪': '%', '،': ',', '٫': '.' 
    }
    trans_table = str.maketrans(arabic_to_western)
    return str(text).translate(trans_table)

def evaluate_chartqa(ans, *gts):
    """Robust ChartQA evaluation handling English, Arabic, visual tolerance, the Year trap, and multiple GT columns."""
    
    ans_norm = normalize_arabic_text(ans)
    ans_clean = str(ans_norm).lower().strip()
    
    # Dynamically gather all valid labels from however many columns are passed
    valid_labels = []
    for gt in gts:
        if pd.isna(gt) or gt is None: # Safely ignore nulls if a column is empty
            continue
        if isinstance(gt, list):
            valid_labels.extend(gt)
        else:
            valid_labels.append(str(gt))
            
    # Iterate through all gathered ground truths
    for label in valid_labels:
        label_norm = normalize_arabic_text(label)
        label_clean = str(label_norm).lower().strip()
        
        # 1. Exact String Match
        if ans_clean == label_clean:
            return True
            
        # 2. Numeric Checks
        ans_num_str = re.sub(r'[\$\£\€\,\%]', '', ans_clean)
        gt_num_str = re.sub(r'[\$\£\€\,\%]', '', label_clean)
        
        try:
            ans_float = float(ans_num_str)
            gt_float = float(gt_num_str)
            
            # --- THE FIX: The Year Trap ---
            # If the ground truth is an integer that looks like a year, force exact match.
            if gt_float.is_integer() and 1800 <= gt_float <= 2100:
                if ans_float == gt_float:
                    return True
                # If it doesn't match exactly, we skip the 5% block below
            else:
                # 5% visual estimation tolerance check for normal data values
                if gt_float == 0:
                    if ans_float == 0: 
                        return True
                elif abs(ans_float - gt_float) / abs(gt_float) <= 0.05:
                    return True
        except ValueError:
            pass 

        # 3. Safe Substring Match 
        if label_clean.isdigit():
            if re.search(fr'\b{label_clean}\b', ans_clean):
                return True
        else:
            if label_clean in ans_clean:
                return True
                
    return False


## Original

In [ ]:
# ─── Load ChartQA dataset ───
chartqa_df = None
if CHARTQA_PATH.exists():
    chartqa_df = pd.read_parquet(CHARTQA_PATH)
    print(f"✅ Loaded {len(chartqa_df)} rows.")
    print(f"Columns: {list(chartqa_df.columns)}")
else:
    print(f"❌ Dataset not found at {CHARTQA_PATH}.")

In [ ]:
# ─── ChartQA run variables ───
CHARTQA_OPENROUTER_LIMIT = None # Set to None for full evaluation

CHARTQA_SYSTEM_PROMPT = """
Answer using ONLY chart data.
RULES:
- Output ONLY the raw final answer. Zero filler or reasoning.
- Numbers: Use digits + units/currencies if applicable (e.g., 45%, $120). Do not spell out.
- If the question asks a yes/no question: output strictly "Yes" or "No".
- Lists: Comma-separated (e.g., A, B, C).
- Labels: Exact text match from the image.

"""

### OpenRouter

In [ ]:
def run_chartqa_openrouter(df, limit=None):
    """Run ChartQA evaluation through OpenRouter backend."""
    print(f"🚀 ChartQA OpenRouter — {MODEL_NAME}...")

    results_dir = Path("ChartQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"chartqa_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["imgname", "query"])

    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break

        imgname  = str(row["imgname"])
        query    = str(row["query"])
        label    = row["label"]
        qa_type  = str(row["type"])
        task_key = f"{imgname}_{query}"

        if task_key in processed:
            continue

        print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

        b64 = encode_image_bytes_to_base64(row["image"])
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{CHARTQA_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_chartqa(answer, label)

        file_exists = append_result({
            "id": str(index), "imgname": imgname, "query": query,
            "ground_truth": str(label), "type": qa_type,
            "model_answer": answer, "evaluation_passed": passed,
            "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_key)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
        time.sleep(1)

# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_openrouter(chartqa_df, limit=CHARTQA_OPENROUTER_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

## Arabic

In [ ]:
# ─── Load ChartQA dataset ───
chartqa_df = None
if AR_CHARTQA_PATH.exists():
    chartqa_df = pd.read_parquet(AR_CHARTQA_PATH)
    chartqa_df['arabic_ground_truth'] = chartqa_df['arabic_ground_truth'].astype(str)
    print(f"✅ Loaded {len(chartqa_df)} rows.")
    print(f"Columns: {list(chartqa_df.columns)}")
else:
    print(f"❌ Dataset not found at {AR_CHARTQA_PATH}.")

In [ ]:
# ─── ChartQA run variables ───
AR_CHARTQA_OPENROUTER_LIMIT = None # Set to None for full evaluation

AR_CHARTQA_SYSTEM_PROMPT = """
أجب باللغة بالعربية و باستخدام بيانات الرسم البياني فقط.
قواعد:
- اكتب الإجابة النهائية المجردة فقط. بدون حشو أو تفسير.
- الأرقام: استخدم الأرقام + الوحدات/العملات إن وجدت (مثل 45%، $120). لا تكتبها حروفاً.
- "إذا كان السؤال يتطلب إجابة بنعم أو لا فقط: أخرج بدقة "نعم" أو "لا.
- القوائم: افصل بينها بفاصلة (مثل أ، ب، ج).
- التسميات: نسخ دقيق للنص كما يظهر في الصورة.

"""

### OpenRouter

In [ ]:
def run_chartqa_openrouter(df, limit=None):
    """Run ChartQA evaluation through OpenRouter backend."""
    print(f"🚀 ChartQA OpenRouter — {MODEL_NAME}...")

    results_dir = Path("ChartQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"chartqaarabic_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["imgname", "query"])

    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break

        imgname  = str(row["imgname"])
        query    = str(row["arabic_query"])
        label    = row["label"]
        label_ar = row["arabic_ground_truth"]
        qa_type  = str(row["type"])
        task_key = f"{imgname}_{query}"

        if task_key in processed:
            continue

        print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

        b64 = encode_image_bytes_to_base64(row["image"])
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{AR_CHARTQA_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_chartqa(answer, label, label_ar)

        file_exists = append_result({
            "id": str(index), "imgname": imgname, "query": query, "ground_truth": str(label),
            "ground_truth_ar": str(label_ar), "type": qa_type,
            "model_answer": answer, "evaluation_passed": passed,
            "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_key)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
        time.sleep(1)

# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_openrouter(chartqa_df, limit=AR_CHARTQA_OPENROUTER_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

# TurtleBench

### Prompt & Dataset Loader

In [ ]:
TURTLEBENCH_SYSTEM_PROMPT = """
INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution.
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""

def load_turtlebench(dataset_path):
    """Parse the official jsonl manifest and load exactly the 260 defined tasks."""
    jsonl_path = Path(dataset_path) / "dataset.jsonl"
    tasks_dir = Path(dataset_path) / "Tasks"
    
    if not jsonl_path.exists():
        print(f"❌ Cannot find {jsonl_path}")
        return pd.DataFrame()

    rows = []
    with open(jsonl_path, 'r', encoding="utf-8-sig") as f:
        for line in f:
            data = json.loads(line)
            
            # Extract directly from the JSON row
            task_id = str(data["id"])
            question_id = f"q{data['question_number']}"
            
            # The input image remains the base shape inside the task folder
            image_path = tasks_dir / task_id / "image" / f"{task_id}.png"
            
            query_text = data.get("query", "")
            variables_text = data.get("variables", "")
            ground_truth_code = data.get("base_shape_code", "No code found.")
            
            # Append critical geometry variables to the prompt
            if variables_text:
                query_text += f"\n\nConstraint - Use these variables strictly:\n{variables_text}"

            rows.append({
                "task_id": task_id,
                "question_id": question_id,
                "image_path": str(image_path),
                "query_text": query_text,
                "ground_truth_code": ground_truth_code
            })
                
    return pd.DataFrame(rows)

### OpenRouter

In [ ]:
# ─── TurtleBench run variables (OpenRouter) ───
TURTLEBENCH_OPENROUTER_LIMIT = None  # Set to None for full evaluation

def run_turtlebench_openrouter(df, limit=None):
    """Run TurtleBench code generation via OpenRouter and save to CSV."""
    print(f"\n🚀 TurtleBench Generation — {MODEL_NAME}...")

    results_dir = Path("TurtleBenchResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    
    safe_name = safe_model_filename(MODEL_NAME)
    output_csv = results_dir / f"turtlebench_{safe_name}_generations.csv"
    
    # Safely load processed keys by combining task and question IDs
    processed_keys = set()
    file_exists = output_csv.exists()
    if file_exists:
        try:
            exist_df = pd.read_csv(output_csv, encoding="utf-8-sig")
            for _, r in exist_df.iterrows():
                processed_keys.add(f"{r['task_id']}_{r['question_id']}")
        except Exception:
            pass

    count = 0  
    skipped_processed_count = 0  # <-- Initialize already-processed counter

    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} questions reached.")
            break

        task_id = str(row["task_id"])
        question_id = str(row["question_id"])
        task_key = f"{task_id}_{question_id}" 

        # Skip if this exact task + question combo is already in the CSV
        if task_key in processed_keys:
            skipped_processed_count += 1
            continue

        # Print how many we skipped right before we start a new one
        if skipped_processed_count > 0:
            print(f"⏭️ Skipped {skipped_processed_count} already-processed tasks.")
            skipped_processed_count = 0 # Reset counter

        print(f"[{index}] Task: {task_id} | Q: {question_id}...")

        query_text = str(row["query_text"])
        image_path = str(row["image_path"])
        ground_truth = str(row["ground_truth_code"])

        b64_url = encode_image_file_to_base64(image_path)
        if not b64_url:
            print("   ⚠️ Skipping — image error.")
            continue

        # PROMPT TUNING
        prompt = f"{TURTLEBENCH_SYSTEM_PROMPT}\n\nTASK:\n{query_text}"
        model_response, gen_time = query_openrouter(prompt, b64_url)

        # Dictionary exactly matches requested columns
        result_dict = {
            "task_id": task_id, 
            "question_id": question_id,
            "image_path": image_path,
            "query_text": query_text,
            "ground_truth_code": ground_truth,
            "model_response": model_response,
            "code_generation_time": round(gen_time, 2)
        }
        
        enc = "utf-8-sig" if not file_exists else "utf-8"
        pd.DataFrame([result_dict]).to_csv(
            output_csv, mode="a", header=not file_exists, index=False,
            encoding=enc,
        )
        file_exists = True
        
        processed_keys.add(task_key)
        count += 1  
        print(f"   ✅ Saved | {gen_time:.2f}s | Task: {task_id} ({question_id})")
        time.sleep(1)

# ─── Execute ───
if TURTLEBENCH_PATH:
    _tb_df = load_turtlebench(TURTLEBENCH_PATH)
    if _tb_df is not None and not _tb_df.empty:
        print(f"✅ Loaded {len(_tb_df)} questions.")
        run_turtlebench_openrouter(_tb_df, limit=TURTLEBENCH_OPENROUTER_LIMIT)
    else:
        print("⚠️ No TurtleBench questions found.")
else:
    print("❌ DTurtleBench folder not found.")

# PEARL

### Prompt & Evaluator

In [ ]:
PEARL_SYSTEM_PROMPT = """
أجب باستخدام الصورة والنص فقط.
أخرج الإجابة الدقيقة فقط. بدون أي حشو أو تفسيرات.
يجب أن تتطابق لغة الإخراج ديناميكيًا مع لغة السؤال تمامًا.
الاختيار من متعدد: أخرج النص الكامل للإجابة، وليس الحرف.
صواب/خطأ: أخرج 'TRUE' أو 'FALSE' فقط.
"""

TRUE_VALUES = {
    "true", "t", "yes", "y", "1", "correct",
    "صواب", "صحيح", "نعم", "صح", "حقيقي", "نعم صحيح",
}
FALSE_VALUES = {
    "false", "f", "no", "n", "0", "incorrect",
    "خطا", "خطأ", "خطاء", "غير صحيح", "لا", "باطل", "غلط",
}


def normalize_arabic_for_eval(text):
    text = str(text).strip().lower()
    text = text.strip("'\"`“”‘’ ")
    arabic_digits = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
    text = text.translate(arabic_digits)
    text = re.sub(r"[\u064b-\u065f\u0670\u0640]", "", text)
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي")
    text = text.replace("ة", "ه")
    text = text.replace("،", ",").replace("٫", ".").replace("٪", "%")
    text = re.sub(r"[\[\]{}()<>:;،,!?؟。،؛]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def strip_option_prefix(text):
    text = normalize_arabic_for_eval(text)
    text = re.sub(r"^(?:option|choice|answer|ans|الاختيار|الخيار|الاجابه|الإجابه)\s*", "", text)
    text = re.sub(r"^[a-d]\s*[\.\)\-:：]\s*", "", text, flags=re.IGNORECASE)
    return text.strip()


def extract_option_letter(text):
    raw = normalize_arabic_for_eval(text).upper()
    raw = raw.strip(" .)-:")
    patterns = [
        r"^([A-D])$",
        r"^([A-D])\s*[\.\)\-:：].*$",
        r"^(?:OPTION|CHOICE|ANSWER|ANS)\s*([A-D])\b.*$",
        r"^(?:الاختيار|الخيار|الاجابه|الإجابه)\s*([A-D])\b.*$",
    ]
    for pattern in patterns:
        match = re.match(pattern, raw, flags=re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None


def normalize_bool_value(text):
    norm = normalize_arabic_for_eval(text)
    norm = strip_option_prefix(norm)
    tokens = set(norm.split())

    if norm in TRUE_VALUES or tokens & TRUE_VALUES:
        return True
    if norm in FALSE_VALUES or tokens & FALSE_VALUES:
        return False

    # Common model formats like "صواب/خطأ: TRUE" or "Answer: FALSE".
    if re.search(r"\btrue\b", norm):
        return True
    if re.search(r"\bfalse\b", norm):
        return False
    return None


def pearl_anls_score(ans, gt, threshold=0.5):
    ans_norm = strip_option_prefix(ans)
    gt_norm = strip_option_prefix(gt)
    if not ans_norm or not gt_norm:
        return 1.0 if ans_norm == gt_norm else 0.0
    if ans_norm == gt_norm:
        return 1.0

    dist = Levenshtein.distance(ans_norm, gt_norm)
    max_len = max(len(ans_norm), len(gt_norm))
    normalized_dist = dist / max_len
    return round(1.0 - normalized_dist, 3) if normalized_dist < threshold else 0.0


def evaluate_pearl_answer(model_answer, gt_text, gt_letter=None, question_type=None, threshold=0.5):
    qtype = normalize_arabic_for_eval(question_type)
    ans = "" if pd.isna(model_answer) else str(model_answer)
    gt = "" if pd.isna(gt_text) else str(gt_text)
    letter = "" if gt_letter is None or pd.isna(gt_letter) else str(gt_letter).strip().upper()

    if qtype == "true_false":
        ans_bool = normalize_bool_value(ans)
        gt_bool = normalize_bool_value(gt)
        if gt_bool is None and letter:
            gt_bool = normalize_bool_value(letter)
        if ans_bool is not None and gt_bool is not None:
            return (1.0 if ans_bool == gt_bool else 0.0), "true_false_exact"
        return pearl_anls_score(ans, gt, threshold), "true_false_anls"

    if qtype == "multiple_choice":
        ans_letter = extract_option_letter(ans)
        if letter and ans_letter:
            return (1.0 if ans_letter == letter else 0.0), "mc_letter"

        text_score = pearl_anls_score(ans, gt, threshold)
        if text_score == 1.0:
            return text_score, "mc_text_exact"
        if text_score > 0:
            return text_score, "mc_text_anls"

        if letter:
            letter_score = pearl_anls_score(ans, letter, threshold)
            if letter_score > 0:
                return letter_score, "mc_letter_anls"
        return 0.0, "mc_no_match"

    # Fallback for unexpected types.
    text_score = pearl_anls_score(ans, gt, threshold)
    return text_score, "text_anls"


def evaluate_pearl_anls(ans, gt_text, gt_letter, threshold=0.5):
    """Compatibility wrapper for old calls."""
    score, _ = evaluate_pearl_answer(ans, gt_text, gt_letter, question_type=None, threshold=threshold)
    return score


### OpenRouter

In [ ]:
PEARL_OPENROUTER_LIMIT = None   # Set to None for full evaluation
PEARL_BATCH_SIZE = 50   # Memory-safe batch size


def run_pearl_openrouter(parquet_path, limit=None):
    """Run PEARL evaluation through OpenRouter using row_id-based resume."""
    if not parquet_path.exists():
        print(f"❌ Dataset not found at {parquet_path}")
        return

    print(f"\n🚀 PEARL OpenRouter — {MODEL_NAME}...")

    results_dir = Path("PEARLResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"pearl_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"

    # Resume by unique parquet row_id, not annotation_id. annotation_id is not unique in PEARL.
    processed, file_exists = load_progress(output_csv, ["row_id"])

    if not file_exists:
        headers = [
            "row_id", "question", "choices", "answer", "answer_letter", "category",
            "country", "image_id", "question_type", "annotation_id",
            "model_answer", "Evaluation", "EvalMethod", "run_time"
        ]
        pd.DataFrame(columns=headers).to_csv(output_csv, index=False, encoding="utf-8-sig")
        file_exists = True

    count = 0
    limit_reached = False
    row_offset = 0
    parquet_file = pq.ParquetFile(parquet_path)

    for batch in parquet_file.iter_batches(batch_size=PEARL_BATCH_SIZE):
        if limit_reached:
            break

        chunk_df = batch.to_pandas()

        if "original_image" in chunk_df.columns:
            chunk_df = chunk_df.rename(columns={"original_image": "image"})

        for local_idx, row in chunk_df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached.")
                limit_reached = True
                break

            row_id = str(row_offset + local_idx)
            task_id = str(row.get("annotation_id", "N/A"))

            if row_id in processed:
                continue

            query         = str(row.get("question", "")).strip()
            gt            = str(row.get("answer", "")).strip()
            ans_letter    = str(row.get("answer_letter", "")).strip()
            category      = str(row.get("category", "N/A"))
            country       = str(row.get("country", "N/A"))
            image_id      = str(row.get("image_id", "N/A"))
            question_type = str(row.get("question_type", "N/A"))

            choices_raw = row.get("choices", "")
            choices_str = ""

            if isinstance(choices_raw, (list, np.ndarray)):
                choices_str = "\n".join([str(c) for c in choices_raw])
            else:
                choices_str = str(choices_raw).strip()
                if choices_str.startswith("[") and choices_str.endswith("]"):
                    try:
                        parsed_list = ast.literal_eval(choices_str)
                        choices_str = "\n".join([str(c) for c in parsed_list])
                    except (ValueError, SyntaxError):
                        pass

            print(f"\nRow {row_id} | Task {task_id} | image_id: {image_id} | type: {question_type}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_image_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            if question_type == "multiple_choice" and choices_str and choices_str not in ("[]", "nan"):
                prompt = f"{query}\n\nOptions:\n{choices_str}\n\n{PEARL_SYSTEM_PROMPT}"
            else:
                prompt = f"{query}\n\n{PEARL_SYSTEM_PROMPT}"

            answer, dur = query_openrouter(prompt, b64)
            score, eval_method = evaluate_pearl_answer(answer, gt, ans_letter, question_type)

            result_dict = {
                "row_id": row_id,
                "question": query,
                "choices": str(choices_raw),
                "answer": gt,
                "answer_letter": ans_letter,
                "category": category,
                "country": country,
                "image_id": image_id,
                "question_type": question_type,
                "annotation_id": task_id,
                "model_answer": answer,
                "Evaluation": score,
                "EvalMethod": eval_method,
                "run_time": round(dur, 2),
            }

            file_exists = append_result(result_dict, output_csv, file_exists)
            processed.add(row_id)
            count += 1

            print(f"   Score: {score:.3f} | {dur:.2f}s | Ans: '{answer}' (GT: '{gt}' / '{ans_letter}')")
            time.sleep(1)

        row_offset += len(chunk_df)


if PEARL_PATH:
    run_pearl_openrouter(PEARL_PATH, limit=PEARL_OPENROUTER_LIMIT)
else:
    print("❌ PEARL_PATH not defined — check your paths.")


# JEEM

### Prompts

In [ ]:
JEEM_GENERATION_PROMPT = """
أجب عن الأسئلة المرفقة بناءً على الصورة فقط، باستخدام اللهجة: {dialect}.
أخرج إجاباتك بصيغة JSON حصراً باستخدام هذه المفاتيح: {expected_keys}.
يجب أن يحتوي الـ JSON على الإجابات فقط بدون أي نص إضافي، أو مقدمات، أو شروحات.

مثال للتنسيق المطلوب:
{{
  "q1": "إجابة السؤال الأول هنا",
  "q3": "إجابة السؤال الثالث هنا"
}}

"""


### OpenRouter

In [ ]:
JEEM_BATCH_SIZE = 20
JEEM_LIMIT = None  # Set to an integer (e.g., 5) to test a small run first

def run_jeem_generation_batched(parquet_path, limit=None):
    if not parquet_path.exists():
        print(f"❌ Dataset not found at {parquet_path}")
        return

    print(f"\n🚀 JEEM Batched Generation — {MODEL_NAME}...")

    results_dir = Path("JEEMResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"jeem_gen_{safe_model_filename(MODEL_NAME)}.csv"
    
    # Track progress using 'id'
    processed, file_exists = load_progress(output_csv, ["id"])

    # Pre-initialize the file to protect Arabic encoding
    if not file_exists:
        # --- UPDATED: Added combined_ground_truth to headers ---
        headers = ["id", "dialect", "combined_questions", "combined_ground_truth", "combined_model_answers", "run_time"]
        pd.DataFrame(columns=headers).to_csv(output_csv, index=False, encoding="utf-8-sig")
        file_exists = True

    count = 0
    limit_reached = False
    parquet_file = pq.ParquetFile(parquet_path)
    
    # Memory-safe batch streaming
    for batch in parquet_file.iter_batches(batch_size=JEEM_BATCH_SIZE):
        if limit_reached: 
            break
            
        chunk_df = batch.to_pandas()
        
        # Map the image column safely for the helper function
        if "original_image" in chunk_df.columns:
            chunk_df = chunk_df.rename(columns={"original_image": "image"})
        
        for _, row in chunk_df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached.")
                limit_reached = True
                break

            task_id = str(row.get("id", "N/A"))
            if task_id in processed:
                continue

            dialect = str(row.get("dialect", "Modern Standard Arabic")).strip()

            # --- 1. Filter Questions & Extract Ground Truth based on 'no_answer' flag ---
            valid_qs = {}
            valid_gts = {}  # --- NEW: Dictionary to hold valid ground truths ---
            for i in range(1, 6):
                no_ans_val = str(row.get(f"no_answer{i}", "")).strip().lower()
                # Skip if the column value indicates it's unanswerable
                is_no_answer = no_ans_val in ['true', '1', 'yes']
                
                if not is_no_answer:
                    q_text = str(row.get(f"question{i}", "")).strip()
                    gt_text = str(row.get(f"answer{i}", "")).strip()  # --- NEW: Grab the correct answer ---
                    
                    if q_text and q_text != "nan":
                        valid_qs[f"q{i}"] = q_text
                        valid_gts[f"q{i}"] = gt_text

            # If all 5 questions were marked 'no_answer', skip calling the model
            if not valid_qs:
                print(f"⏭️ Task {task_id} skipped (all questions marked as no_answer).")
                processed.add(task_id)
                continue

            # --- 2. Format the Token-Optimized Prompt ---
            expected_keys_str = ", ".join([f'"{ k}"' for k in valid_qs.keys()])
            qs_text = "\n".join([f"سؤال {k.replace('q','')}: {v}" for k, v in valid_qs.items()])
            
            sys_prompt = JEEM_GENERATION_PROMPT.format(
                dialect=dialect, 
                expected_keys=expected_keys_str
            )
            final_prompt = f"{sys_prompt}\n\nالأسئلة:\n{qs_text}"

            # --- 3. Extract and Process Image ---
            img_bytes = extract_image_bytes(row)
            b64 = encode_image_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print(f"⚠️ Skipping Task {task_id} — image error.")
                continue

            # --- 4. Query the Model ---
            print(f"\nTask {task_id} | Dialect: {dialect} | Valid Questions: {len(valid_qs)}/5")
            model_response, dur = query_openrouter(final_prompt, b64)
            
            # --- 5. Parse JSON & Combine Answers ---
            model_answers_dict = {}
            try:
                # Strip markdown blocks safely
                clean_json = model_response.strip("` \n").replace("json\n", "")
                model_answers_dict = json.loads(clean_json)
            except Exception as e:
                print(f"   ⚠️ JSON Parse Error: {e}")
                # Fallback to save raw text to avoid losing the generation
                model_answers_dict = {k: f"[PARSE ERROR] Raw: {model_response}" for k in valid_qs.keys()}

            # Combine valid questions, ground truth, and model answers with Q/A prefixes
            combined_questions = "\n".join([f"Q{k.replace('q','')}: {v}" for k, v in valid_qs.items()])
            
            # --- UPDATED: Combine the ground truth answers ---
            combined_ground_truth = "\n".join([f"A{k.replace('q','')}: {v}" for k, v in valid_gts.items()])
            
            combined_answers = "\n".join([f"A{k.replace('q','')}: {model_answers_dict.get(k, 'MISSING')}" for k in valid_qs.keys()])

            # --- 6. Save the Results ---
            result_dict = {
                "id": task_id,
                "dialect": dialect,
                "combined_questions": combined_questions,
                "combined_ground_truth": combined_ground_truth,  # --- NEW: Added to final dict ---
                "combined_model_answers": combined_answers,
                "run_time": round(dur, 2)
            }
            
            file_exists = append_result(result_dict, output_csv, file_exists)
            processed.add(task_id)
            count += 1
            
            print(f"   ✅ Saved {len(valid_qs)} answers | {dur:.2f}s")
            time.sleep(1)

# --- Execution ---
if JEEM_PATH:
    run_jeem_generation_batched(JEEM_PATH, limit=JEEM_LIMIT)
else:
    print("❌ JEEM_PATH not defined.")

# ScreenQA

### Prompt & Evaluator

In [ ]:
# SCREENQA_SYSTEM_PROMPT = """
# Answer using ONLY the screenshot.
# 1. EXACT MATCH: Copy UI text exactly (case/punctuation/spelling). No synonyms.
# 2. NUMBERS: Output digits (e.g., "4") unless explicitly spelled out on screen.
# 3. Output ONLY "Yes" or "No" for boolean queries.
# 4. Output ONLY the exact answer. Zero explanations or filler.
# """

# # ─── Image resize dimensions for ScreenQA ───
# SCREENQA_RESIZE = (1280, 720)


# def evaluate_screenqa(ans, gt, threshold=0.5):
#     """Calculates ANLS (Average Normalized Levenshtein Similarity) for ScreenQA."""
#     # 1. Clean strings (lowercase, strip whitespace, remove currencies)
#     ans_str = re.sub(r'[\$\£\€\,]', '', str(ans).lower()).strip()
#     gt_str  = re.sub(r'[\$\£\€\,]', '', str(gt).lower()).strip()

#     # 2. Handle empty strings safely
#     if not ans_str or not gt_str:
#         return 1.0 if ans_str == gt_str else 0.0

#     # 3. Calculate Levenshtein Distance
#     dist = Levenshtein.distance(ans_str, gt_str)
    
#     # 4. Normalize by the maximum length
#     max_len = max(len(ans_str), len(gt_str))
#     normalized_dist = dist / max_len

#     # 5. Apply the threshold (default 0.5)
#     # If the error rate is under 50%, give partial credit. Otherwise, 0 score.
#     if normalized_dist < threshold:
#         return round(1.0 - normalized_dist, 2)
#     else:
#         return 0.0

# def extract_screenqa_ground_truth(row):
#     """
#     Robust extractor using Regex to find the short 'text' answer within 
#     nested, stringified, or NumPy-wrapped structures.
#     """
#     # 1. Pull the raw data and force it into a string format for searching
#     raw_str = str(row.get("answers", row.get("ground_truth", row.get("full_answer", ""))))

#     # 2. PRIORITY: Extract the specific UI element text (e.g., "$70", "Grace", "Free")
#     # This looks for 'text': followed by a quote, and captures everything inside the quotes.
#     text_match = re.search(r"'text':\s*['\"]([^'\"]+)['\"]", raw_str)
#     if text_match:
#         return text_match.group(1).strip()

#     # 3. FALLBACK: Extract the 'full_answer' conversational string if 'text' isn't found
#     full_match = re.search(r"'full_answer':\s*['\"]([^'\"]+)['\"]", raw_str)
#     if full_match:
#         return full_match.group(1).strip()

#     # 4. FINAL FALLBACK: Return the cleaned raw string
#     return raw_str.strip()

### OpenRouter

In [ ]:
# SCREENQA_OPENROUTER_LIMIT = 2  # Set to None for full evaluation
# SCREENQA_BATCH_SIZE = 50  # Number of rows to load into RAM at a time

# def run_screenqa_openrouter(parquet_path, limit=None):
#     """Run ScreenQA evaluation through OpenRouter using memory-safe chunking."""
#     if not parquet_path.exists():
#         print(f"❌ Dataset not found at {parquet_path}.")
#         return

#     print(f"\n🚀 ScreenQA OpenRouter — {MODEL_NAME}...")

#     results_dir = Path("ScreenQAResults")
#     results_dir.mkdir(parents=True, exist_ok=True)
#     output_csv = results_dir / f"screenqa_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    
#     # Load progress to resume safely
#     processed, file_exists = load_progress(output_csv, ["id"])

#     # Open the Parquet file as a stream rather than loading it all
#     parquet_file = pq.ParquetFile(parquet_path)
#     print(f"📊 Total dataset rows: {parquet_file.metadata.num_rows}")
    
#     count = 0
#     limit_reached = False
    
#     # Loop through the dataset in safe batches
#     for batch_idx, batch in enumerate(parquet_file.iter_batches(batch_size=SCREENQA_BATCH_SIZE)):
#         if limit_reached:
#             break
            
#         print(f"\n📦 Loading Batch {batch_idx + 1} into RAM...")
#         chunk_df = batch.to_pandas()
        
#         for _, row in chunk_df.iterrows():
#             # Check limit
#             if limit and count >= limit:
#                 print(f"🎯 Limit of {limit} reached.")
#                 limit_reached = True
#                 break

#             task_id = str(row["screen_id"])
            
#             # Skip already processed
#             if task_id in processed:
#                 continue

#             image_name = str(row.get("file_name", "N/A"))
#             query = str(row["question"])
#             gt = extract_screenqa_ground_truth(row)

#             print(f"\nTask {task_id} | Image: {image_name}")

#             img_bytes = extract_image_bytes(row)
#             b64 = encode_image_bytes_to_base64(img_bytes, resize=SCREENQA_RESIZE) if img_bytes else None
#             if not b64:
#                 print("   ⚠️ Skipping — image error.")
#                 continue

#             prompt = f"{query}\n{SCREENQA_SYSTEM_PROMPT}"
#             answer, dur = query_openrouter(prompt, b64)
            
#             # Calculate ANLS Score (using the integrated function from earlier)
#             passed = evaluate_screenqa(answer, gt)

#             file_exists = append_result({
#                 "id": task_id, "image": image_name, "question": query,
#                 "ground_truth": gt, "model_answer": answer,
#                 "evaluation_passed": passed, "run_time": round(dur, 2),
#             }, output_csv, file_exists)

#             processed.add(task_id)
#             count += 1
            
#             # If using ANLS, 'passed' will be a float score. Format it nicely.
#             score_display = f"{passed:.3f}" if isinstance(passed, float) else ('✅' if passed else '❌')
#             print(f"   Score: {score_display} | {dur:.2f}s | Ans: '{answer}' (GT: '{gt}')")
            
#             time.sleep(1)
            
#         # Once this loop finishes, `chunk_df` is garbage collected. RAM is instantly freed!

# # Notice we are passing the PATH directly now, not the pre-loaded dataframe
# if SCREENQA_PATH:
#     run_screenqa_openrouter(SCREENQA_PATH, limit=SCREENQA_OPENROUTER_LIMIT)
# else:
#     print("❌ SCREENQA_PATH is not defined.")

# Results Evaluation

In [6]:
JUDGE_MODEL = "google/gemini-3.1-flash-lite-preview"

### ChartQA Eval

In [ ]:
from datetime import datetime

CHARTQA_RESULTS_DIR = Path("ChartQAResults")
CHARTQA_NUMERIC_TOLERANCE = 0.05
CHARTQA_CREATE_BACKUP = False


def chartqa_eval_normalize_text(value):
    """Normalize text for ChartQA answer comparison, including Arabic digits/punctuation."""
    if pd.isna(value):
        return ""

    text = normalize_arabic_text(str(value)).lower().strip()
    text = text.replace("\u00a0", " ").replace("\u202f", " ")
    text = re.sub(r"[`*_~\"'،؛:;!?()\[\]{}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def chartqa_eval_compact_number_text(value):
    """Remove separators inside digits so values like '40 040' and '40,040' become '40040'."""
    text = normalize_arabic_text(str(value)).lower()
    text = text.replace("\u00a0", " ").replace("\u202f", " ")
    previous = None
    while previous != text:
        previous = text
        text = re.sub(r"(?<=\d)[\s,_](?=\d)", "", text)
    return text


def chartqa_eval_extract_labels(*values):
    """Collect valid ground-truth labels from scalar, list-like, or bracketed string cells."""
    labels = []
    for value in values:
        if value is None or pd.isna(value):
            continue
        if isinstance(value, (list, tuple, set, np.ndarray)):
            labels.extend(value)
            continue

        text = str(value).strip()
        if not text or text.lower() == "nan":
            continue

        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, (list, tuple, set)):
                    labels.extend(parsed)
                    continue
            except Exception:
                labels.extend(part.strip() for part in text.strip("[]").split(",") if part.strip())
                continue

        labels.append(text)
    return [str(label).strip() for label in labels if str(label).strip()]


def chartqa_eval_numbers(value):
    """Extract comparable numeric values after fixing grouped digits and simple percent/currency formatting."""
    text = chartqa_eval_compact_number_text(value)
    text = re.sub(r"[^0-9.\-+% ]", " ", text)
    numbers = []
    for match in re.finditer(r"[-+]?\d+(?:\.\d+)?%?", text):
        raw = match.group(0).rstrip("%")
        try:
            numbers.append(float(raw))
        except ValueError:
            pass
    return numbers


def chartqa_eval_numeric_match(model_answer, ground_truth, tolerance=CHARTQA_NUMERIC_TOLERANCE):
    ans_nums = chartqa_eval_numbers(model_answer)
    gt_nums = chartqa_eval_numbers(ground_truth)
    if not ans_nums or not gt_nums:
        return False

    for gt_num in gt_nums:
        for ans_num in ans_nums:
            if gt_num == ans_num:
                return True

            # Years should match exactly; a 5% tolerance is too loose for dates.
            if float(gt_num).is_integer() and 1800 <= gt_num <= 2100:
                continue

            if gt_num == 0:
                if ans_num == 0:
                    return True
            elif abs(ans_num - gt_num) / abs(gt_num) <= tolerance:
                return True
    return False


def chartqa_eval_string_match(model_answer, ground_truth):
    ans = chartqa_eval_normalize_text(model_answer)
    gt = chartqa_eval_normalize_text(ground_truth)
    if not ans or not gt:
        return False

    if ans == gt:
        return True

    ans_compact = chartqa_eval_compact_number_text(ans)
    gt_compact = chartqa_eval_compact_number_text(gt)
    if ans_compact == gt_compact:
        return True

    # Allow full-label containment for textual answers, but avoid very short accidental matches.
    if not re.fullmatch(r"[-+]?\d+(?:\.\d+)?", gt_compact):
        if len(gt) >= 3 and re.search(rf"(?<!\w){re.escape(gt)}(?!\w)", ans):
            return True
    return False


def evaluate_chartqa_robust(model_answer, *ground_truth_values):
    """Re-evaluate ChartQA rows with stronger numeric grouping, Arabic, list, and text handling."""
    labels = chartqa_eval_extract_labels(*ground_truth_values)
    for label in labels:
        if chartqa_eval_string_match(model_answer, label):
            return True
        if chartqa_eval_numeric_match(model_answer, label):
            return True
    return False


def chartqa_is_false(value):
    return str(value).strip().lower() in {"false", "0", "0.0", "no", "nan", ""}


def revaluate_chartqa_false_rows(results_dir=CHARTQA_RESULTS_DIR):
    """Re-read every ChartQA CSV and update only rows currently marked False in evaluation_passed."""
    results_dir = Path(results_dir)
    csv_files = sorted(results_dir.glob("*.csv"))
    print(f"Found {len(csv_files)} ChartQA result CSV files in {results_dir}.")

    summary = []
    backup_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding="utf-8-sig")
        required = {"model_answer", "evaluation_passed"}
        if not required.issubset(df.columns) or "ground_truth" not in df.columns:
            print(f"Skipping {csv_file.name}: missing required columns.")
            continue

        false_mask = df["evaluation_passed"].apply(chartqa_is_false)
        false_before = int(false_mask.sum())
        fixed_count = 0

        for idx in df.index[false_mask]:
            row = df.loc[idx]
            gt_values = [row.get("ground_truth")]
            if "ground_truth_ar" in df.columns:
                gt_values.append(row.get("ground_truth_ar"))

            if evaluate_chartqa_robust(row.get("model_answer"), *gt_values):
                df.at[idx, "evaluation_passed"] = True
                fixed_count += 1

        if fixed_count:
            if CHARTQA_CREATE_BACKUP:
                backup_file = csv_file.with_suffix(f".before_chartqa_reeval_{backup_stamp}.csv")
                csv_file.replace(backup_file)
                df.to_csv(csv_file, index=False, encoding="utf-8-sig")
            else:
                df.to_csv(csv_file, index=False, encoding="utf-8-sig")

        false_after = false_before - fixed_count
        summary.append({
            "file": csv_file.name,
            "rows": len(df),
            "false_before": false_before,
            "fixed_false_rows": fixed_count,
            "false_after": false_after,
        })
        print(f"{csv_file.name}: fixed {fixed_count} of {false_before} false rows.")

    return pd.DataFrame(summary)


chartqa_reeval_summary = revaluate_chartqa_false_rows()
chartqa_reeval_summary


### MathVision Eval

In [ ]:
TARGET_FOLDER_PATH = "MathVisionResults"
JUDGETEMP = 0.0

from decimal import Decimal, InvalidOperation
from fractions import Fraction


def strip_latex_wrappers(text):
    """Remove common final-answer wrappers without trying to simplify math."""
    text = str(text).strip()
    text = re.sub(r"```(?:\w+)?|```", "", text).strip()
    text = re.sub(r"\\boxed\{([^{}]+)\}", r"\1", text)
    text = re.sub(r"\\text\{([^{}]+)\}", r"\1", text)
    text = text.replace("−", "-").replace("–", "-").replace("—", "-")
    return text.strip()


def normalize_plain_answer(text):
    text = strip_latex_wrappers(text).lower().strip()
    text = re.sub(r"^(?:final\s+answer|answer|ans)\s*(?:is|:)?\s*", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .")


def extract_single_choice(text):
    """Return A-E only when the model answer clearly gives one option, otherwise None."""
    cleaned = normalize_plain_answer(text).upper()
    cleaned = cleaned.strip(" .")

    patterns = [
        r"^[\(\[]?([A-E])[\)\].]?$",
        r"^(?:OPTION|CHOICE)\s*[\(\[]?([A-E])[\)\].]?$",
        r"^(?:THE\s+)?(?:ANSWER|ANS|FINAL\s+ANSWER)\s*(?:IS|:)?\s*[\(\[]?([A-E])[\)\].]?$",
    ]
    for pattern in patterns:
        match = re.fullmatch(pattern, cleaned)
        if match:
            return match.group(1)
    return None


def compact_numeric_text(text):
    text = strip_latex_wrappers(text).lower()
    text = text.replace(",", "")
    text = re.sub(r"(?<=\d)\s+(?=\d)", "", text)
    text = text.replace("$", "").replace("£", "").replace("€", "")
    return text.strip()


def parse_single_numeric_value(text):
    """Parse one clear numeric/fraction value. Return (value, has_percent) or (None, False)."""
    raw = compact_numeric_text(text)
    raw = re.sub(r"^(?:final\s+answer|answer|ans)\s*(?:is|:)?\s*", "", raw).strip()
    raw = raw.strip(" .")

    # Reject equations, inequalities, coordinate pairs, lists, and expressions. Let the LLM judge these.
    if re.search(r"[=<>]|\bor\b|\band\b|[,;]", raw):
        return None, False

    has_percent = "%" in raw or "percent" in raw or "percentage" in raw
    without_units = re.sub(
        r"\b(?:cm|mm|m|km|kg|g|lbs?|feet|ft|inches|inch|degrees?|units?|dollars?|usd|percent|percentage)\b",
        " ",
        raw,
    )
    without_units = without_units.replace("%", " ")
    without_units = re.sub(r"\s+", " ", without_units).strip()

    fraction_matches = re.findall(r"[-+]?\d+\s*/\s*[-+]?\d+", without_units)
    decimal_matches = re.findall(r"[-+]?(?:\d+\.\d+|\d+|\.\d+)", without_units)

    if len(fraction_matches) == 1 and not re.sub(r"[-+]?\d+\s*/\s*[-+]?\d+", "", without_units).strip():
        try:
            return Decimal(Fraction(fraction_matches[0].replace(" ", "")).numerator) / Decimal(Fraction(fraction_matches[0].replace(" ", "")).denominator), has_percent
        except Exception:
            return None, False

    if len(decimal_matches) == 1 and not re.sub(r"[-+]?(?:\d+\.\d+|\d+|\.\d+)", "", without_units).strip():
        try:
            return Decimal(decimal_matches[0]), has_percent
        except InvalidOperation:
            return None, False

    return None, False


def numeric_values_match(ans, gt):
    ans_value, ans_percent = parse_single_numeric_value(ans)
    gt_value, gt_percent = parse_single_numeric_value(gt)
    if ans_value is None or gt_value is None:
        return None

    candidates = [(ans_value, gt_value)]
    if ans_percent and not gt_percent:
        candidates.append((ans_value / Decimal("100"), gt_value))
    if gt_percent and not ans_percent:
        candidates.append((ans_value, gt_value / Decimal("100")))

    for left, right in candidates:
        tolerance = max(abs(right) * Decimal("1e-9"), Decimal("1e-9"))
        if abs(left - right) <= tolerance:
            return True
    return False


def looks_complex_or_ambiguous(ans, gt):
    combined = f"{ans} {gt}".lower()
    if re.search(r"\\|\^|=|sqrt|√|pi|π|\(|\)|\[|\]|\{|\}|±|≈|≤|≥|<|>|∠|°", combined):
        return True
    # Multiple numbers often require context, so let the judge decide.
    number_count = len(re.findall(r"[-+]?(?:\d+\.\d+|\d+|\.\d+)(?:\s*/\s*\d+)?", compact_numeric_text(ans)))
    return number_count > 1


def llm_math_judge_openrouter(question, model_answer, ground_truth):
    """Strict compact LLM fallback for ambiguous MathVision equivalence."""
    if not JUDGE_MODEL:
        print("   ⚠️ JUDGE_MODEL is not set; ambiguous row marked False.")
        return False, "judge_missing"

    prompt = (
        "Grade answer equivalence for a math/image QA item. "
        "Return True only if the model answer is clearly equivalent to GT in the question context; "
        "for multiple choice, the selected option must match. If unsure, return False. "
        "Reply only True or False.\n"
        f"Q: {question}\nGT: {ground_truth}\nA: {model_answer}"
    )

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": JUDGE_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": JUDGETEMP,
    }

    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers, json=payload, timeout=120,
        )
        response.raise_for_status()
        result_text = response.json()["choices"][0]["message"]["content"].strip().lower()
        if result_text == "true":
            return True, "llm_judge"
        if result_text == "false":
            return False, "llm_judge"
        return False, "judge_parse_error"
    except Exception as e:
        print(f"   ⚠️ OpenRouter Judge Error: {e}")
        return False, "judge_error"


def evaluate_mathvision_row(question, ans, gt):
    """Conservative layered evaluation: easy certainties locally, ambiguous cases to LLM."""
    ans_norm = normalize_plain_answer(ans)
    gt_norm = normalize_plain_answer(gt)

    if not ans_norm or ans_norm in {"nan", "none", "error_max_retries_reached"}:
        return False, "empty_or_error_answer"

    if ans_norm == gt_norm:
        return True, "exact"

    gt_choice = extract_single_choice(gt_norm)
    if gt_choice:
        ans_choice = extract_single_choice(ans_norm)
        if ans_choice:
            return ans_choice == gt_choice, "multiple_choice"
        return llm_math_judge_openrouter(question, ans, gt)

    numeric_match = numeric_values_match(ans, gt)
    if numeric_match is not None:
        return numeric_match, "numeric"

    if looks_complex_or_ambiguous(ans, gt):
        return llm_math_judge_openrouter(question, ans, gt)

    return False, "no_safe_match"


def grade_results_csv(csv_path):
    print(f"\n🚀 Evaluating {csv_path.name} using conservative checks + {JUDGE_MODEL} fallback...")

    try:
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
    except Exception as e:
        print(f"❌ Error reading file {csv_path.name}: {e}")
        return

    if not all(col in df.columns for col in ["question", "model_answer", "ground_truth"]):
        print(f"❌ CSV {csv_path.name} is missing required columns ('question', 'model_answer', 'ground_truth'). Skipping.")
        return

    judgments = []
    methods = []
    for row in df.itertuples():
        is_correct, method = evaluate_mathvision_row(
            str(row.question), str(row.model_answer), str(row.ground_truth)
        )
        judgments.append(is_correct)
        methods.append(method)

        mark = "✅" if is_correct else "❌"
        print(f"Row {row.Index} | {mark} | {method} | Model: {str(row.model_answer)[:25]} | GT: {str(row.ground_truth)[:25]}")

    df["ComplexEval"] = judgments
    df["EvalMethod"] = methods
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    accuracy = (sum(judgments) / len(judgments)) * 100 if judgments else 0
    print(f"\n📊 Final Accuracy for {csv_path.name}: {accuracy:.2f}%")
    print(f"💾 Updated in place: {csv_path.name}")


def process_folder():
    folder = Path(TARGET_FOLDER_PATH)
    if not folder.exists() or not folder.is_dir():
        print(f"❌ Folder not found: {TARGET_FOLDER_PATH}")
        return

    csv_files = [
        f for f in folder.glob("*.csv")
        if ".before_" not in f.name and not f.name.startswith("~")
    ]

    if not csv_files:
        print(f"⚠️ No CSV files found in '{TARGET_FOLDER_PATH}'.")
        return

    print(f"📂 Found {len(csv_files)} CSV file(s) to process in '{TARGET_FOLDER_PATH}'.")
    for csv_file in sorted(csv_files):
        grade_results_csv(csv_file)

    print("\n🎉 All files processed successfully!")


if __name__ == "__main__":
    process_folder()


### TurtleBench Eval

In [ ]:
RESULTS_FOLDER = "TurtleBenchResults"

# Hybrid TurtleBench evaluation:
# 1. Run deterministic visual tracing/raster IoU first.
# 2. If visual evaluation is True, copy True into the final judge result.
# 3. If visual evaluation is False/error, send only that row to the LLM judge.

import math
import re
import types
from PIL import Image, ImageDraw
import numpy as np

TURTLE_VISUAL_IOU_THRESHOLD = 0.95
TURTLE_VISUAL_CANVAS_SIZE = 512
TURTLE_VISUAL_PADDING = 24
TURTLE_VISUAL_LINE_WIDTH = 3
TURTLE_VISUAL_ARC_STEPS = 96

TURTLEBENCH_EVAL_PROMPT_TEMPLATE = """
Judge whether two Python Turtle code snippets draw the same final visible shape.
Be lenient about setup: ignore imports, turtle object creation, screen setup, speed, pen color, comments, and harmless penup/goto positioning used only to place the same shape.
Focus on geometry: visible lines/arcs/circles, relative lengths, angles, and final drawn output. Order matters only if it changes the visible drawing.
Return exactly this format:
True - one short reason
or
False - one short reason

Model code:
{model_code}

Ground truth code:
{ground_truth_code}
"""


class FakeTurtle:
    def __init__(self):
        self.x = 0.0
        self.y = 0.0
        self.heading_deg = 0.0
        self.is_down = True
        self.strokes = []

    def _point(self):
        return (self.x, self.y)

    def _line_to(self, x, y):
        start = self._point()
        end = (float(x), float(y))
        if self.is_down and start != end:
            self.strokes.append((start, end))
        self.x, self.y = end

    def forward(self, distance):
        rad = math.radians(self.heading_deg)
        self._line_to(self.x + float(distance) * math.cos(rad), self.y + float(distance) * math.sin(rad))

    def backward(self, distance):
        self.forward(-float(distance))

    def back(self, distance):
        self.backward(distance)

    def left(self, angle):
        self.heading_deg = (self.heading_deg + float(angle)) % 360

    def right(self, angle):
        self.heading_deg = (self.heading_deg - float(angle)) % 360

    def setheading(self, angle):
        self.heading_deg = float(angle) % 360

    def heading(self):
        return self.heading_deg

    def goto(self, x, y=None):
        if y is None:
            x, y = x
        self._line_to(x, y)

    def setpos(self, x, y=None):
        self.goto(x, y)

    def setposition(self, x, y=None):
        self.goto(x, y)

    def home(self):
        self.goto(0, 0)
        self.setheading(0)

    def penup(self):
        self.is_down = False

    def pendown(self):
        self.is_down = True

    def up(self):
        self.penup()

    def down(self):
        self.pendown()

    def isdown(self):
        return self.is_down

    def pos(self):
        return self._point()

    def position(self):
        return self._point()

    def xcor(self):
        return self.x

    def ycor(self):
        return self.y

    def circle(self, radius, extent=None, steps=None):
        radius = float(radius)
        if radius == 0:
            return
        extent = 360.0 if extent is None else float(extent)
        steps = int(steps or max(8, min(TURTLE_VISUAL_ARC_STEPS, abs(extent) / 360 * TURTLE_VISUAL_ARC_STEPS)))

        heading = math.radians(self.heading_deg)
        left_normal = (-math.sin(heading), math.cos(heading))
        center = (self.x + left_normal[0] * radius, self.y + left_normal[1] * radius)
        start_angle = math.atan2(self.y - center[1], self.x - center[0])
        direction = 1 if radius > 0 else -1

        prev = self._point()
        for i in range(1, steps + 1):
            theta = start_angle + direction * math.radians(extent) * (i / steps)
            point = (center[0] + abs(radius) * math.cos(theta), center[1] + abs(radius) * math.sin(theta))
            if self.is_down and prev != point:
                self.strokes.append((prev, point))
            prev = point

        self.x, self.y = prev
        self.heading_deg = (self.heading_deg + extent) % 360

    # Style/setup no-ops.
    def speed(self, *args, **kwargs): pass
    def color(self, *args, **kwargs): pass
    def pencolor(self, *args, **kwargs): pass
    def fillcolor(self, *args, **kwargs): pass
    def pensize(self, *args, **kwargs): pass
    def width(self, *args, **kwargs): pass
    def hideturtle(self, *args, **kwargs): pass
    def showturtle(self, *args, **kwargs): pass
    def begin_fill(self, *args, **kwargs): pass
    def end_fill(self, *args, **kwargs): pass
    def dot(self, size=5, *args, **kwargs):
        r = float(size) / 2
        x, y = self._point()
        self.strokes.extend([((x-r, y), (x+r, y)), ((x, y-r), (x, y+r))])


class FakeTurtleModule:
    def __init__(self, turtle_obj):
        self._turtle = turtle_obj

    def Turtle(self):
        return self._turtle

    def Screen(self):
        return types.SimpleNamespace(setup=lambda *a, **k: None, tracer=lambda *a, **k: None, update=lambda *a, **k: None)

    def __getattr__(self, name):
        if hasattr(self._turtle, name):
            return getattr(self._turtle, name)
        return lambda *args, **kwargs: None


def strip_code_fences(code):
    text = str(code).strip()
    text = re.sub(r"^```(?:python|py)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.replace("```python", "").replace("```py", "").replace("```", "")
    return text.strip()


def extract_turtle_variables(query_text):
    text = str(query_text)
    if "Constraint - Use these variables strictly:" in text:
        text = text.split("Constraint - Use these variables strictly:", 1)[1]

    assignment_lines = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or "=" not in line:
            continue
        if re.match(r"^[A-Za-z_]\w*\s*=\s*[-+*/().\w\s]+$", line):
            assignment_lines.append(line)

    if not assignment_lines:
        return {}

    env = {
        "__builtins__": {"abs": abs, "min": min, "max": max, "round": round, "int": int, "float": float},
        "math": math,
    }
    try:
        exec("\n".join(assignment_lines), env, env)
    except Exception:
        return {}

    return {
        key: value for key, value in env.items()
        if not key.startswith("__") and key != "math" and isinstance(value, (int, float, str, tuple, list))
    }


def safe_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name == "math":
        return math
    if name == "turtle":
        return globals.get("turtle") if globals and "turtle" in globals else None
    raise ImportError(f"Import blocked: {name}")


def trace_turtle_code(code, variables=None):
    clean_code = strip_code_fences(code)
    t = FakeTurtle()
    turtle_module = FakeTurtleModule(t)
    safe_builtins = {
        "abs": abs, "min": min, "max": max, "sum": sum, "round": round,
        "len": len, "range": range, "int": int, "float": float,
        "list": list, "tuple": tuple, "enumerate": enumerate,
        "__import__": safe_import,
    }
    env = {
        "__builtins__": safe_builtins,
        "math": math,
        "turtle": turtle_module,
        "t": t,
    }
    if variables:
        env.update(variables)

    try:
        exec(clean_code, env, env)
        return {"strokes": t.strokes, "error": None}
    except Exception as e:
        return {"strokes": t.strokes, "error": str(e)}


def rasterize_strokes(strokes, size=TURTLE_VISUAL_CANVAS_SIZE, padding=TURTLE_VISUAL_PADDING, line_width=TURTLE_VISUAL_LINE_WIDTH):
    img = Image.new("L", (size, size), 0)
    if not strokes:
        return np.array(img, dtype=bool)

    points = [p for stroke in strokes for p in stroke]
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    width = max(max_x - min_x, 1e-9)
    height = max(max_y - min_y, 1e-9)
    scale = (size - 2 * padding) / max(width, height)

    def transform(point):
        x, y = point
        tx = padding + (x - min_x) * scale + ((size - 2 * padding) - width * scale) / 2
        ty = padding + (max_y - y) * scale + ((size - 2 * padding) - height * scale) / 2
        return (tx, ty)

    draw = ImageDraw.Draw(img)
    for start, end in strokes:
        draw.line([transform(start), transform(end)], fill=255, width=line_width)
    return np.array(img, dtype=bool)


def mask_iou(mask_a, mask_b):
    union = np.logical_or(mask_a, mask_b).sum()
    if union == 0:
        return 1.0
    intersection = np.logical_and(mask_a, mask_b).sum()
    return float(intersection / union)


def evaluate_turtle_visual_equivalence(model_code, gt_code, query_text="", threshold=TURTLE_VISUAL_IOU_THRESHOLD):
    variables = extract_turtle_variables(query_text)
    model_trace = trace_turtle_code(model_code, variables=variables)
    gt_trace = trace_turtle_code(gt_code, variables=variables)

    if model_trace["error"]:
        return False, 0.0, f"model_exec_error: {model_trace['error']}"
    if gt_trace["error"]:
        return False, 0.0, f"gt_exec_error: {gt_trace['error']}"
    if not model_trace["strokes"] or not gt_trace["strokes"]:
        return False, 0.0, "missing_drawn_strokes"

    model_mask = rasterize_strokes(model_trace["strokes"])
    gt_mask = rasterize_strokes(gt_trace["strokes"])
    score = mask_iou(model_mask, gt_mask)
    return score >= threshold, score, f"visual_iou={score:.3f}; vars={sorted(variables.keys())}"


def parse_turtle_judge_response(response_text):
    text = str(response_text).strip()
    match = re.match(r"^(true|false)\b\s*(?:[-:–—]\s*)?(.*)$", text, flags=re.IGNORECASE | re.DOTALL)
    if not match:
        return "ERROR", text
    verdict = match.group(1).lower() == "true"
    reason = re.sub(r"\s+", " ", match.group(2).strip())
    return verdict, reason or "No explanation provided."


def evaluate_turtle_equivalence_openrouter(model_code, gt_code):
    prompt = TURTLEBENCH_EVAL_PROMPT_TEMPLATE.format(
        model_code=strip_code_fences(model_code),
        ground_truth_code=strip_code_fences(gt_code),
    )

    for attempt in range(MAX_RETRIES):
        try:
            response_text, _ = query_openrouter_text(prompt, model_name=JUDGE_MODEL)
            return parse_turtle_judge_response(response_text)
        except Exception as e:
            print(f"   ❌ Eval attempt {attempt + 1}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(5 * (attempt + 1))

    return "ERROR", "Max retries reached."


def evaluate_all_results(results_dir):
    target_dir = Path(results_dir)
    if not target_dir.exists():
        print(f"❌ Directory not found: {target_dir}")
        return

    csvs_to_process = sorted(f for f in target_dir.glob("*.csv") if not f.stem.endswith("_LLM_JUDGED"))
    if not csvs_to_process:
        print("✅ No generation CSVs found to evaluate.")
        return

    print(f"🚀 Starting hybrid TurtleBench evaluation on {len(csvs_to_process)} file(s)...")

    for csv_path in csvs_to_process:
        print(f"\n📄 Processing: {csv_path.name}")
        df = pd.read_csv(csv_path, encoding="utf-8-sig")

        if not all(col in df.columns for col in ["model_response", "ground_truth_code"]):
            print(f"⚠️ Skipping {csv_path.name}: missing model_response or ground_truth_code.")
            continue

        visual_results = []
        visual_scores = []
        visual_reasons = []
        final_results = []
        final_reasons = []
        eval_methods = []

        for row in df.itertuples():
            task_id = getattr(row, "task_id", f"Row_{row.Index}")
            question_id = getattr(row, "question_id", "")
            model_code = getattr(row, "model_response", "")
            gt_code = getattr(row, "ground_truth_code", "")
            query_text = next(
                (getattr(row, col, "") for col in ("query", "query_text", "question", "prompt") if str(getattr(row, col, "")).strip()),
                ""
            )

            print(f"Evaluating Task {task_id} {question_id}...")

            if not str(model_code).strip() or not str(gt_code).strip() or str(model_code).lower() == "nan" or str(gt_code).lower() == "nan":
                visual_results.append(False)
                visual_scores.append(0.0)
                visual_reasons.append("missing_code")
                final_results.append(False)
                final_reasons.append("Missing code for comparison.")
                eval_methods.append("missing_code")
                continue

            visual_passed, visual_score, visual_reason = evaluate_turtle_visual_equivalence(
                model_code, gt_code, query_text
            )
            visual_results.append(visual_passed)
            visual_scores.append(round(visual_score, 4))
            visual_reasons.append(visual_reason)

            if visual_passed is True:
                final_results.append(True)
                final_reasons.append(visual_reason)
                eval_methods.append("visual")
                print(f"   ✅ Visual pass | {visual_reason}")
                continue

            llm_result, llm_reason = evaluate_turtle_equivalence_openrouter(model_code, gt_code)
            final_results.append(llm_result)
            final_reasons.append(llm_reason)
            eval_methods.append("llm_after_visual_false")
            print(f"   {'✅' if llm_result is True else '❌' if llm_result is False else '⚠️'} LLM judge after visual fail: {llm_result}")
            time.sleep(1)

        df["VisualEval_Result"] = visual_results
        df["VisualEval_IoU"] = visual_scores
        df["VisualEval_Reason"] = visual_reasons
        df["LLM_Judge_Result"] = final_results
        df["LLM_Judge_Reasoning"] = final_reasons
        df["EvalMethod"] = eval_methods
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"💾 Updated in place: {csv_path.name}")


evaluate_all_results(
    results_dir=RESULTS_FOLDER
)


### JEEM Eval

In [ ]:
TARGET_FOLDER = Path("JEEMResults")
JUDGE_MAX_RETRIES = 3
JEEM_QUESTION_KEYS = [f"q{i}" for i in range(1, 6)]

JEEM_JUDGE_PROMPT = """
أنت مقيم دلالي منصف لإجابات أسئلة صور باللهجة: {dialect}.
قارن كل إجابة نموذج بالإجابة الصحيحة لنفس رقم السؤال فقط.

احكم TRUE عندما تحمل إجابة النموذج نفس المعنى الأساسي أو تجيب عن السؤال بشكل صحيح، حتى لو كانت:
- أقصر أو أطول من الإجابة الصحيحة.
- مكتوبة بلهجة عربية مختلفة أو فصحى أو بمرادفات.
- فيها اختلافات بسيطة في التشكيل، الهمزات، التاء المربوطة، الأرقام، أو علامات الترقيم.
- تستخدم نعم/لا، اه/لأ، صح/خطأ، أو صياغة مختصرة لمعنى واضح.

احكم FALSE فقط إذا كان المعنى مخالفاً، أو يضيف معلومة أساسية خاطئة، أو لا يجيب عن السؤال.
لا تعاقب على اختلاف الأسلوب أو اللهجة. لا تستخدم معلومات خارج النص إلا لفهم المرادفات الواضحة.
قيّم فقط الأسئلة الموجودة في الطلب. إذا لم يوجد q5 مثلاً فلا تحسبه في total_q.

أخرج JSON صحيحاً فقط، بدون Markdown أو شرح، بهذا الشكل:
{{
  "q1": true,
  "q2": false,
  "q3": true,
  "correct_q": 2,
  "total_q": 3,
  "score": 0.6667
}}
"""


def normalize_jeem_text(value):
    """Normalize Arabic/English answer text for high-confidence exact matching."""
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    if text in {"", "nan", "none", "null"}:
        return ""

    arabic_digit_map = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
    text = text.translate(arabic_digit_map)
    text = re.sub(r"[\u064b-\u065f\u0670\u0640]", "", text)
    text = re.sub(r"[إأآٱا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[^\w\s\u0600-\u06ff]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_jeem_bool(value):
    text = normalize_jeem_text(value)
    if not text:
        return None

    yes_values = {
        "نعم", "اه", "اي", "ايوه", "ايوا", "اها", "صح", "صحيح", "مبين", "يوجد", "في", "yes", "true"
    }
    no_values = {
        "لا", "لاء", "لأ", "كلا", "مش", "مو", "ما", "مفيش", "غير مبين", "مو مبين", "مش مبين", "no", "false"
    }

    tokens = set(text.split())
    if text in yes_values or tokens & yes_values:
        if not (tokens & {"لا", "مش", "مو", "ما", "مفيش", "غير"}):
            return True
    if text in no_values or tokens & no_values:
        return False
    return None


def parse_jeem_json_like(text):
    if pd.isna(text):
        return {}
    raw = str(text).strip()
    if not raw or raw.lower() in {"nan", "none", "null"}:
        return {}

    clean = raw.strip("` \n")
    clean = re.sub(r"^json\s*", "", clean, flags=re.IGNORECASE).strip()
    try:
        data = json.loads(clean)
        if isinstance(data, dict):
            return {
                str(k).lower().strip(): "" if pd.isna(v) else str(v).strip()
                for k, v in data.items()
                if re.fullmatch(r"q?[1-5]|a?[1-5]", str(k).lower().strip())
            }
    except Exception:
        pass
    return {}


def parse_jeem_qa_block(text, prefix):
    """Parse Q1/A1 style blocks into {'q1': text, ...}."""
    parsed = parse_jeem_json_like(text)
    if parsed:
        normalized = {}
        for key, value in parsed.items():
            number = re.search(r"[1-5]", key)
            if number:
                normalized[f"q{number.group(0)}"] = value
        return normalized

    if pd.isna(text):
        return {}
    raw = str(text)
    if not raw.strip() or raw.strip().lower() in {"nan", "none", "null"}:
        return {}

    pattern = re.compile(
        rf"(?:^|\n)\s*{prefix}\s*([1-5])\s*[:：\-]\s*(.*?)(?=\n\s*{prefix}\s*[1-5]\s*[:：\-]|$)",
        flags=re.IGNORECASE | re.DOTALL,
    )
    return {f"q{num}": ans.strip() for num, ans in pattern.findall(raw) if ans.strip()}


def jeem_easy_match(model_answer, ground_truth):
    """Return True for safe deterministic matches, otherwise None for LLM judging."""
    model_norm = normalize_jeem_text(model_answer)
    gt_norm = normalize_jeem_text(ground_truth)

    if not model_norm or not gt_norm:
        return False if not model_norm and gt_norm else None

    if model_norm == gt_norm:
        return True

    model_bool = normalize_jeem_bool(model_answer)
    gt_bool = normalize_jeem_bool(ground_truth)
    if model_bool is not None and gt_bool is not None:
        return model_bool == gt_bool

    model_number = re.fullmatch(r"-?\d+(?:\.\d+)?", model_norm)
    gt_number = re.fullmatch(r"-?\d+(?:\.\d+)?", gt_norm)
    if model_number and gt_number:
        return float(model_norm) == float(gt_norm)

    if len(gt_norm.split()) <= 3 and (model_norm in gt_norm or gt_norm in model_norm):
        return True

    return None


def clean_jeem_judge_json(response):
    clean = str(response).strip()
    clean = re.sub(r"^```(?:json)?\s*", "", clean, flags=re.IGNORECASE)
    clean = re.sub(r"\s*```$", "", clean).strip()
    match = re.search(r"\{.*\}", clean, flags=re.DOTALL)
    if match:
        clean = match.group(0)
    clean = re.sub(r",\s*([}\]])", r"\1", clean)
    return json.loads(clean)


def build_jeem_judge_prompt(dialect, question_items, ground_truth_items, model_answer_items):
    judge_sys_prompt = JEEM_JUDGE_PROMPT.format(dialect=dialect)
    questions = "\n".join(f"{key.upper()}: {question_items.get(key, '')}" for key in question_items)
    ground_truth = "\n".join(f"{key.upper()}: {ground_truth_items.get(key, '')}" for key in question_items)
    model_answers = "\n".join(f"{key.upper()}: {model_answer_items.get(key, '')}" for key in question_items)
    return f"{judge_sys_prompt}\n\nالأسئلة:\n{questions}\n\nالإجابات الصحيحة:\n{ground_truth}\n\nإجابات النموذج:\n{model_answers}"


def evaluate_jeem_row(row):
    dialect = str(row.get("dialect", "Modern Standard Arabic"))
    questions = parse_jeem_qa_block(row.get("combined_questions", ""), "Q")
    ground_truth = parse_jeem_qa_block(row.get("combined_ground_truth", ""), "A")
    model_answers = parse_jeem_qa_block(row.get("combined_model_answers", ""), "A")

    valid_keys = [key for key in JEEM_QUESTION_KEYS if normalize_jeem_text(ground_truth.get(key, ""))]
    if not valid_keys:
        return {**{key: False for key in JEEM_QUESTION_KEYS}, "correct_q": 0, "total_q": 0, "score": 0.0, "judge_run_time": 0.0}

    results = {}
    unresolved_keys = []
    for key in valid_keys:
        easy = jeem_easy_match(model_answers.get(key, ""), ground_truth.get(key, ""))
        if easy is None:
            unresolved_keys.append(key)
        else:
            results[key] = bool(easy)

    judge_run_time = 0.0
    if unresolved_keys:
        prompt = build_jeem_judge_prompt(
            dialect=dialect,
            question_items={key: questions.get(key, "") for key in unresolved_keys},
            ground_truth_items={key: ground_truth.get(key, "") for key in unresolved_keys},
            model_answer_items={key: model_answers.get(key, "") for key in unresolved_keys},
        )
        response, judge_run_time = query_openrouter_text(prompt, model_name=JUDGE_MODEL, max_retries=JUDGE_MAX_RETRIES)
        try:
            judge_data = clean_jeem_judge_json(response)
            for key in unresolved_keys:
                results[key] = bool(judge_data.get(key, False))
        except Exception as e:
            print(f"   ⚠️ Judge JSON Parse Error: {e}\nRaw Response: {response}")
            for key in unresolved_keys:
                results[key] = False

    correct_q = sum(1 for key in valid_keys if results.get(key) is True)
    total_q = len(valid_keys)
    score = correct_q / total_q if total_q else 0.0

    output = {key: (results.get(key, False) if key in valid_keys else "") for key in JEEM_QUESTION_KEYS}
    output.update({
        "correct_q": correct_q,
        "total_q": total_q,
        "score": round(score, 4),
        "judge_run_time": round(judge_run_time, 2),
    })
    return output


def runfolder_evaluation():
    if not TARGET_FOLDER.exists():
        print(f"❌ Target folder {TARGET_FOLDER} does not exist.")
        return

    csv_files = [f for f in TARGET_FOLDER.glob("*.csv") if not f.name.startswith("eval_")]
    if not csv_files:
        print(f"⚠️ No raw generation CSVs found to evaluate in {TARGET_FOLDER}.")
        return

    print(f"📂 Found {len(csv_files)} original CSV files to evaluate using {JUDGE_MODEL}.")

    metric_cols = JEEM_QUESTION_KEYS + ["correct_q", "total_q", "score", "judge_run_time"]

    for csv_path in csv_files:
        print(f"\n⚖️ Evaluating file: {csv_path.name}")
        output_csv = TARGET_FOLDER / f"eval_{csv_path.name}"
        raw_df = pd.read_csv(csv_path, encoding="utf-8-sig")

        if output_csv.exists():
            eval_df = pd.read_csv(output_csv, encoding="utf-8-sig")
            if "id" in eval_df.columns:
                eval_df = eval_df.drop_duplicates(subset=["id"], keep="last").set_index("id", drop=False)
            else:
                eval_df = pd.DataFrame().set_index(pd.Index([], name="id"))
        else:
            eval_df = pd.DataFrame().set_index(pd.Index([], name="id"))

        updated_rows = []
        for _, row in raw_df.iterrows():
            task_id = str(row["id"])
            existing = eval_df.loc[task_id].to_dict() if task_id in eval_df.index else None

            already_done = False
            if existing is not None:
                already_done = all(col in existing and not pd.isna(existing[col]) for col in ["correct_q", "total_q", "score"])

            if already_done:
                result_dict = existing
            else:
                print(f"   Task {task_id} | Evaluating...")
                result_dict = row.to_dict()
                result_dict.update(evaluate_jeem_row(row))
                print(f"   ✅ Score: {result_dict['correct_q']}/{result_dict['total_q']} ({result_dict['score']:.2%})")
                if result_dict.get("judge_run_time", 0) > 0:
                    time.sleep(1)

            updated_rows.append(result_dict)

        output_df = pd.DataFrame(updated_rows)
        for col in metric_cols:
            if col not in output_df.columns:
                output_df[col] = "" if col in JEEM_QUESTION_KEYS else 0

        base_cols = [col for col in raw_df.columns if col in output_df.columns]
        ordered_cols = base_cols + [col for col in metric_cols if col in output_df.columns]
        extra_cols = [col for col in output_df.columns if col not in ordered_cols]
        output_df = output_df[ordered_cols + extra_cols]
        output_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
        print(f"💾 Updated in place: {output_csv.name}")

    print("\n🎉 JEEM evaluation complete!")


# --- Execution ---
runfolder_evaluation()


📂 Found 5 original CSV files to evaluate using google/gemini-3.1-flash-lite-preview.

⚖️ Evaluating file: jeem_gen_openai_gpt-4o-mini.csv
   Task 0d230acc6d848fb1a4872364c22ba072 | Evaluating...
   ✅ Score: 2/5 (40.00%)
   Task d63a178f2540792acecf47c11941d2b9 | Evaluating...
   ✅ Score: 4/5 (80.00%)
   Task 2cee8d74f0d95a22e5be72c4fcb918f8 | Evaluating...
   ✅ Score: 3/5 (60.00%)
   Task 78f3acc64aa1b8abf54e1563beb13b8b | Evaluating...
   ✅ Score: 3/5 (60.00%)
   Task 28fd039de0d30f787f12f9b5435fb0b9 | Evaluating...
   ✅ Score: 4/5 (80.00%)
   Task dca18c7fc06bb8b8f2d705c3a25e1c20 | Evaluating...
   ✅ Score: 2/5 (40.00%)
   Task 9e3463e12da30fd21769f1b0a23e8e12 | Evaluating...
   ✅ Score: 1/5 (20.00%)
   Task 3530270dc16efb9a7fdc53df44b942c3 | Evaluating...
   ✅ Score: 3/5 (60.00%)
   Task a9e5c53d19f0812756c34e7176440715 | Evaluating...
   ✅ Score: 3/5 (60.00%)
   Task 4d00aeec79f0fcbe5ad1a60a30ffc50f | Evaluating...
   ✅ Score: 3/5 (60.00%)
   Task fdd0dafb1e75decbc2f4291093ec4e49 |

### Final Eval

In [ ]:
# ─── Folders to scan for result CSVs ───
RESULTS_FOLDERS = [
    "MathVisionResults", 
    "ChartQAResults",
    "TurtleBenchResults", 
    "JEEMResults", 
    "PEARLResults"
]

# Add any dataset-specific evaluation column names here.
# The first matching column found in each CSV will be used.
EVAL_COLUMN_CANDIDATES = [
    "evaluation_passed",
    "Evaluation",
    "ComplexEval",
    "LLM_Judge_Result",
    "score",
    "is_perfect",
]


def normalize_column_name(name):
    """Normalize column names so checks are case/spacing/underscore-insensitive."""
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())


def find_eval_column(df, candidates=EVAL_COLUMN_CANDIDATES):
    """Return the real CSV column name matching one of the configured candidates."""
    normalized_columns = {normalize_column_name(col): col for col in df.columns}
    for candidate in candidates:
        match = normalized_columns.get(normalize_column_name(candidate))
        if match:
            return match
    return None


def coerce_eval_score(value):
    """Convert bools, numeric scores, and fraction strings into a 0-1 score."""
    if pd.isna(value):
        return 0.0
    if isinstance(value, bool):
        return 1.0 if value else 0.0
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip().lower()
    if text in {"true", "yes", "pass", "passed", "correct", "equivalent"}:
        return 1.0
    if text in {"false", "no", "fail", "failed", "incorrect", "not equivalent", "", "nan", "none"}:
        return 0.0

    fraction_match = re.search(r"(\d+(?:\.\d+)?)\s*(?:/|out\s*of|outof)\s*(\d+(?:\.\d+)?)", text)
    if fraction_match:
        numerator = float(fraction_match.group(1))
        denominator = float(fraction_match.group(2))
        return numerator / denominator if denominator else 0.0

    try:
        return float(text)
    except ValueError:
        return 0.0


def should_show_threshold_accuracy(eval_col):
    """Return True for 0-1 similarity score columns where >= 0.5 is a match."""
    return normalize_column_name(eval_col) in {
        normalize_column_name("Evaluation"),
    }


def summarize_results(folders=RESULTS_FOLDERS):
    """Print mean metric scores and thresholded accuracy where applicable."""
    header = f"{'Model Name':<60} | {'Rows':>8} | {'Metric Column':<18} | {'Mean Score':>10} | {'>=0.5 Accuracy':>14}"
    print(header)
    print("-" * len(header))
    total_rows = 0

    for folder in folders:
        folder_path = Path(folder)
        if not folder_path.exists():
            continue

        print(f"\n{'=' * 24} {folder} {'=' * 24}")
        for csv_file in sorted(folder_path.glob("*.csv")):
            try:
                df = pd.read_csv(csv_file, encoding="utf-8-sig")
                row_count = len(df)
                if row_count == 0:
                    continue

                eval_col = find_eval_column(df)
                if eval_col is None:
                    print(f"⏭️ Skipping {csv_file.name}: no evaluation column found.")
                    continue

                scores = df[eval_col].apply(coerce_eval_score)
                mean_score = scores.mean() * 100
                threshold_accuracy = (scores >= 0.5).mean() * 100 if should_show_threshold_accuracy(eval_col) else None
                name = csv_file.stem.replace("_openrouter", "").replace("_results", "")
                threshold_display = f"{threshold_accuracy:>12.2f}%" if threshold_accuracy is not None else f"{'N/A':>13}"
                print(f"{name:<60} | {row_count:>8} | {eval_col:<18} | {mean_score:>8.2f}% | {threshold_display}")
                total_rows += row_count
            except Exception as e:
                print(f"⚠️ Error processing {csv_file.name}: {e}")

    print("-" * len(header))
    print(f"{'TOTAL ROWS':<60} | {total_rows:>8}")


summarize_results()

In [ ]:
# Count rows in every CSV across all result folders
RESULTS_FOLDERS = [
    "MathVisionResults", 
    "ChartQAResults",
    "ScreenQAResults",
    "TurtleBenchResults", 
    "JEEMResults", 
    "PEARLResults",
]

overall_total_rows = 0

for folder in RESULTS_FOLDERS:
    folder_path = Path(folder)
    if not folder_path.exists():
        print(f"Skipping {folder}: folder not found.")
        continue

    folder_total_rows = 0
    csv_files = sorted(folder_path.glob("*.csv"))

    print(f"\n{'=' * 24} {folder} {'=' * 24}")
    if not csv_files:
        print("No CSV files found.")
        continue

    for csv_file in csv_files:
        try:
            row_count = len(pd.read_csv(csv_file, encoding="utf-8-sig"))
            folder_total_rows += row_count
            overall_total_rows += row_count
            print(f"{csv_file.name:<80} {row_count:>8}")
        except Exception as e:
            print(f"Error reading {csv_file.name}: {e}")

    print(f"{'Folder total':<80} {folder_total_rows:>8}")

print("\n" + "=" * 96)
print(f"{'OVERALL TOTAL ROWS':<80} {overall_total_rows:>8}")
